# Alasfour and Gilja (2024) on AJILE12 (DANDI:000055)

Paper: A. Alasfour and V. Gilja, "Consistent spectro-spatial features of human ECoG successfully
decode naturalistic behavioral states", Front. Hum. Neurosci. 18:1388267 (2024),
doi:10.3389/fnhum.2024.1388267. The claims file (`claims/alasfour-2024.yaml`) was written from the
version of record (PMC11169785); the paper has no supplement and links no code.

Dataset: DANDI:000055 (AJILE12), version 0.220127.0436, streamed from the archive. The paper
cites no version; this is the only published version (the draft has the same 55 assets and size),
so it is pinned.

Outside input: the electrode-to-ROI projection matrices of Peterson et al. (2021a), Figshare item
doi:10.6084/m9.figshare.12114408.v1, which the claims file names for every claim on regions
(`data: with_inputs`).

| Claim | Tested here |
|---|---|
| roi-projection-link (link claim added by the author) | tested |
| cohort-selection | tested |
| table1-participants | tested |
| fig2-highest-accuracy, fig2-rois-three-bands, fig2-inferior-temporal-all-bands, fig2-beta-temporal-only, fig2-single-band-rois | tested |
| anova-roi-band-effects, anova-no-interaction | tested |
| posthoc-band-averages, posthoc-roi-averages | tested |
| fig3-three-class | tested |
| fig4-not-single-participant, fig4-missing-rois, fig4-participant2-high | tested |
| fig5-gamma-higher-talking, fig5-theta-alpha-lower-talking, fig5-pre-postcentral, fig5-parietal-temporal-no-gamma, fig5-middle-temporal-tv | tested |
| inferior-temporal-high-gamma-inconsistent, all-rois-but-inferior-parietal | tested |
| fig6-sign-consistency, fig6-examples | tested |
| fig7-non-additive, fig7-participant4-low-gamma, fig7-above-70, discussion-above-80, fig7-exceeds-group, fig7-best-band-varies | tested |

Readings. The claims file names readings for details the paper leaves open (O1 to O16). Every
analysis is run under a grid of them: artifact rule O4 (a, b) by SD (i, ii); ROI assignment O5
(a, b, and c, a reading taken from Methods 2.1, see the helper) by density rule (i, ii); ROI feature
O6 (a, b); buffer O7 (a, b); chance level O9, alpha (0.05, 0.01) by n (per fold, per participant).
The main reading, the one the paper best supports in this attempt's judgement, is O4 (a)(ii),
O5 (c)(ii), O6 (a), O7 (a), O9 alpha 0.05 with n all test samples of the participant (Figure 2
draws one dashed chance line). Each claim's section prints the main reading's
result and how many readings of the grid meet the criterion.

In [ ]:
import os
import time

os.environ.setdefault("OMP_NUM_THREADS", "1")
os.environ.setdefault("OPENBLAS_NUM_THREADS", "1")
import itertools
import resource

import numpy as np
import pandas as pd
from scipy import stats
import statsmodels.formula.api as smf
from statsmodels.stats.anova import anova_lm
from statsmodels.stats.multicomp import pairwise_tukeyhsd
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis

import ajile as A

pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 40)
T0 = time.time()
P = A.PARTICIPANTS
SHORT = {r: A.ROI_NAMES[r] for r in A.ROIS}

## Survey of the dataset

Every asset of the pinned version, and the metadata (epochs, electrodes, series attributes) of every
file; no signal data are read here.

In [ ]:
assets = A.list_assets()
print(len(assets), "assets,", assets.subject.nunique(), "subjects,", round(assets["size"].sum() / 1e9, 1), "GB")
print(assets.groupby("subject").session.apply(list).to_string())
META = A.survey(assets)
m0, el0, ep0 = META[("08", 3)]
print("ElectricalSeries:", m0["shape"], "chunks", m0["chunks"], "rate", m0["rate"], "unit", m0["unit"], "conversion", m0["conversion"])
print("electrodes columns:", list(el0.columns))
print("epochs columns: labels, start_time, stop_time; labels seen:")
labs = pd.concat([META[k][2].label for k in META]).value_counts()
print(labs.to_string())

## Feature extraction

For the five participants of Table 1, every block labelled with exactly one of Talk, TV or
Computer/phone is streamed (only the chunks covering the block, plus 5 s each side, and only the
electrodes good on at least one day), filtered into the five bands, enveloped, and averaged into
10-s bins. The artifact statistics of each block are kept for every reading of the artifact rule.
The epochs of two files are on a clock 8.5 h and 8.3 h ahead of the ECoG (data check below); the
offset is applied here.

In [ ]:
CH, JOBS, KEYS = {}, [], []
for p in P:
    days = sorted(s for (q, s) in META if q == p)
    good = np.zeros(len(META[(p, days[0])][1]), bool)
    for d in days:
        good |= META[(p, d)][1].good.values
    ch = np.where(good)[0]
    CH[p] = ch
    nsamp = np.array([META[(p, d)][0]["shape"][0] for d in days])
    sdd = np.array([META[(p, d)][1].standard_deviation.values[ch] for d in days])
    pooled = np.sqrt((nsamp[:, None] * sdd**2).sum(0) / nsamp.sum())
    for i, d in enumerate(days):
        b = A.class_blocks(META[(p, d)][2])
        aid = assets[(assets.subject == p) & (assets.session == d)].asset_id.iloc[0]
        JOBS.append((aid, ch, b, [sdd[i], pooled], A.epoch_offset(META[(p, d)][0])))
        KEYS.append((p, d))
order = np.argsort([-len(j[2]) * len(j[1]) for j in JOBS])
JOBS = [JOBS[i] for i in order]
KEYS = [KEYS[i] for i in order]
t = time.time()
RES = A.extract_participant_days(JOBS, procs=4)
print("extracted", sum(len(j[2]) for j in JOBS), "blocks from", len(JOBS), "files,",
      round(sum(r[1] for r in RES) / 1e9, 1), "GB read, in", round((time.time() - t) / 60, 1), "min")

In [ ]:
DAYS = {p: [] for p in P}
for (p, d), job, r in sorted(zip(KEYS, JOBS, RES), key=lambda z: z[0]):
    m, el, ep = META[(p, d)]
    t0 = pd.Timestamp(m["session_start"]).timestamp() - A.epoch_offset(m)
    DAYS[p].append((d, t0, el.good.values[CH[p]], job[2], r[0]))
print({p: [d[0] for d in DAYS[p]] for p in P})

## Data check

In [ ]:
print("Per file: duration of the ECoG, epochs' last stop, offset applied, class blocks, blocks outside the ECoG")
rows = []
for p in P:
    for d, t0, good, blocks, res in DAYS[p]:
        m = META[(p, d)][0]
        rows.append(dict(participant=p, session=d, ecog_h=m["shape"][0] / m["rate"] / 3600,
                         epochs_end_h=META[(p, d)][2].stop.max() / 3600, offset_h=A.epoch_offset(m) / 3600,
                         start=m["session_start"], blocks=len(blocks), outside=sum(r is None for r in res),
                         good=int(good.sum()), channels_read=len(good)))
FILES = pd.DataFrame(rows)
print(FILES.round(3).to_string())
print("ECoG duration + offset, h: min %.3f, max %.3f" % ((FILES.ecog_h + FILES.offset_h).min(), (FILES.ecog_h + FILES.offset_h).max()))
for k in [("01", 7), ("04", 3)]:
    ep = META[k][2]
    first = ep[~ep.label.str.startswith("Blocklist (Data")].start.min()
    print(k, "first epoch that is not a data break starts at %.1f s; offset %.1f s; ECoG starts at 0 s of its own clock"
          % (first, A.epoch_offset(META[k][0])))
print()
print("Units: ElectricalSeries stored in", m0["unit"], "with conversion", m0["conversion"], "-> values are microvolts")
print("Sampling rate (all files):", sorted({META[k][0]["rate"] for k in META}))
print("Block durations, s:", end=" ")
dur = pd.concat([b.stop - b.start for p in P for _, _, _, b, _ in DAYS[p]])
print("min %.1f, median %.1f, max %.1f, n %d; outside 110 to 130 s: %d; shorter than 10 s (no bin): %d"
      % (dur.min(), dur.median(), dur.max(), len(dur), ((dur < 110) | (dur > 130)).sum(), (dur < 10).sum()))
print()
print("SD column of the electrodes table against the RMS of the class blocks (median ratio RMS / SD per file):")
for p in P:
    rr = []
    for d, t0, good, blocks, res in DAYS[p]:
        ss = np.nansum([r["sumsq"] for r in res if r is not None], axis=0)
        nn = np.sum([np.isfinite(r["sumsq"]) * r["n"] for r in res if r is not None], axis=0)
        sd = META[(p, d)][1].standard_deviation.values[CH[p]]
        rr.append(np.nanmedian(np.sqrt(ss / nn)[good] / sd[good]))
    print(" ", p, np.round(rr, 2))
print()
print("Envelope check against sosfiltfilt + scipy.signal.hilbert on one block and channel:")
r0 = DAYS["08"][0]
aid = assets[(assets.subject == "08") & (assets.session == 3)].asset_id.iloc[0]
f, _ = A.open_nwb(aid)
b = r0[3].iloc[0]
i0, i1 = int(round(b.start * 500)), int(round(b.stop * 500))
x = f["acquisition/ElectricalSeries/data"][max(0, i0 - 2500): i1 + 2500, CH["08"][0]].astype(float)
x = x - np.median(x)
o = i0 - max(0, i0 - 2500)
for bi, ref in enumerate(A.envelope_check(x)):
    rb = np.array([ref[o + k * 5000: o + (k + 1) * 5000].mean() for k in range(r0[4][0]["feats"].shape[0])])
    ours = r0[4][0]["feats"][:, 0, bi]
    print("  %-10s max relative difference over %d bins: %.4f (values %.2f to %.2f uV)"
          % (A.BAND_NAMES[bi], len(rb), np.max(np.abs(ours - rb) / rb), rb.min(), rb.max()))
f.close()

### Readings and samples

The analysed channels are the electrodes good on every day of the participant (an assumption: the
paper gives one count per participant, and a fixed channel set is needed across days). Samples are
the 10-s bins of kept blocks, in time order; a bin with a missing value in an analysed channel is
dropped.

In [ ]:
FILES_FIG = A.figshare_files()[0]
CEN = A.load_centroids(FILES_FIG)
PROJ = {p: A.load_projection(p, FILES_FIG) for p in P}
ART = {"a-ii": ("a", 0), "b-ii": ("b", 0), "a-i": ("a", 1), "b-i": ("b", 1)}
SAMPLES = {}
for an, ar in ART.items():
    for p in P:
        inter = np.all([g for _, _, g, _, _ in DAYS[p]], axis=0)
        S = A.build_samples(DAYS[p], ar)
        SAMPLES[(an, p)] = A.restrict(S, inter)
        SAMPLES[(an, p)]["channels"] = CH[p][inter]
rows = []
for (an, p), S in SAMPLES.items():
    st = S["blocks"].status.value_counts()
    c = pd.Series(S["cls"]).value_counts()
    rows.append(dict(artifact=an, participant=p, channels=len(S["channels"]), blocks_kept=st.get("kept", 0),
                     blocks_artifact=st.get("artifact", 0), blocks_outside=st.get("outside recording", 0),
                     bins_nan_dropped=S["n_nan_dropped"], Talk=c.get("Talk", 0), TV=c.get("TV", 0),
                     Electronics=c.get("Electronics", 0)))
print(pd.DataFrame(rows).to_string())


def roi_sets(p, o5, dens_r, channels):
    dens, W = PROJ[p]
    el = META[(p, DAYS[p][0][0])][1]
    asg = A.roi_assignment(dens, W, el[["x", "y", "z"]].values, CEN, o5)
    return A.roi_channels(asg, dens, channels, dens_r)


GRID = list(itertools.product(ART, ["c", "a", "b"], ["ii", "i"], ["a", "b"], ["a", "b"]))
MAIN = ("a-ii", "c", "ii", "a", "a")
CHANCES = [(0.05, "total"), (0.01, "total"), (0.05, "fold"), (0.01, "fold")]
t = time.time()
FOLDS = {}
for g in GRID:
    an, o5, dr, o6, o7 = g
    parts = []
    for p in P:
        S = SAMPLES[(an, p)]
        rois = roi_sets(p, o5, dr, S["channels"])
        df = A.run_two_class(S, rois, o6, o7, seed=1000 + int(p))
        df["participant"] = p
        parts.append(df)
    FOLDS[g] = A.add_chance(pd.concat(parts, ignore_index=True))
GROUP = {(g, ch): A.group_tests(FOLDS[g], ch[0], ch[1]) for g in GRID for ch in CHANCES}
print(len(GRID), "pipelines,", len(GROUP), "readings with the chance levels;", round(time.time() - t), "s")
print("Main reading:", MAIN)
print("ROIs per participant under the main reading (electrodes):")
for p in P:
    S = SAMPLES[("a-ii", p)]
    print(" ", p, {SHORT[r]: len(v) for r, v in roi_sets(p, "c", "ii", S["channels"]).items()})


def readings_meeting(pred, chance=True):
    """Readings of the grid (and chance levels) under which pred(group table, folds) holds."""
    ok = []
    for g in GRID:
        for ch in (CHANCES if chance else CHANCES[:1]):
            if pred(GROUP[(g, ch)], FOLDS[g]):
                ok.append((g, ch))
    return ok


def show_readings(ok, chance=True):
    n = len(GRID) * (len(CHANCES) if chance else 1)
    main = (MAIN, CHANCES[0]) in ok
    print("criterion met under the main reading:", main, "| readings meeting it: %d of %d" % (len(ok), n))
    if ok:
        df = pd.DataFrame([dict(artifact=g[0], O5=g[1] + "-" + g[2], O6=g[3], O7=g[4], alpha=c[0], n=c[1]) for g, c in ok])
        for col in df.columns:
            print("   ", col, df[col].value_counts().to_dict())
    return main


def mat(G, col, fmt=None):
    m = G.pivot(index="band", columns="roi", values=col).reindex(index=A.BAND_NAMES, columns=A.ROIS)
    m.columns = [SHORT[c] for c in m.columns]
    return m


GM = GROUP[(MAIN, CHANCES[0])]
FM = FOLDS[MAIN]

### Clock check

Decoding Talk against TV with the true labels and with the labels shifted in time (the pooled
samples rolled by half their number), under the main reading, high gamma of all ROI electrodes. For
the two files with an epoch offset, a classifier trained on the participant's other days is tested
on that day, with and without the offset.

In [ ]:
rng = np.random.default_rng(7)
for p in P:
    S = SAMPLES[("a-ii", p)]
    pos = sorted({i for v in roi_sets(p, "c", "ii", S["channels"]).values() for i in v})
    F = S["X"][:, pos, 4].astype(float)
    y = S["cls"]
    out = []
    for shift in [0, len(y) // 2]:
        ys = np.roll(y, shift)
        sp = A.make_splits(ys, ["Talk", "TV"], "a", rng)
        acc, _, nt = A.cv_two_class(F, ys == "Talk", sp)
        out.append(acc.mean())
    print("participant", p, "accuracy true labels %.3f, shifted labels %.3f" % tuple(out))
print("Leave-one-day-out: trained on the participant's other days, tested on one day (high gamma, all ROI electrodes):")
for p in ["01", "04"]:
  for d in sorted(set(SAMPLES[("a-ii", p)]["day"])):
    S = SAMPLES[("a-ii", p)]
    pos = sorted({i for v in roi_sets(p, "c", "ii", S["channels"]).values() for i in v})
    tr = np.where((S["day"] != d) & np.isin(S["cls"], ["Talk", "TV"]))[0]
    te = np.where((S["day"] == d) & np.isin(S["cls"], ["Talk", "TV"]))[0]
    if min((S["cls"][te] == c).sum() for c in ["Talk", "TV"]) < 5:
        print("  participant %s day %d: fewer than 5 bins of a class" % (p, d))
        continue
    tr, te = A.balance(tr, S["cls"], rng), A.balance(te, S["cls"], rng)
    F = S["X"][:, pos, 4].astype(float)
    pred, _, _ = A.lda2(A.zscore(F[tr]), S["cls"][tr] == "Talk", A.zscore(F[te]))
    acc = (pred == (S["cls"][te] == "Talk")).mean()
    print("  participant %s day %d%s: %d balanced bins, accuracy %.3f (finite chance at 0.05: %.3f)"
          % (p, d, " (offset applied)" if (p, d) in [("01", 7), ("04", 3)] else "", len(te), acc, A.finite_chance(len(te), 2, 0.05)))
print("Without the offset, %d and %d of the class blocks of these two files lie past the end of the ECoG."
      % tuple(int(((META[k][2].stop[META[k][2].label.isin(list(A.CLASS_LABELS))]) * 500 > META[k][0]["shape"][0]).sum())
              for k in [("01", 7), ("04", 3)]))

## roi-projection-link (link claim)

"The projection matrices of Peterson et al. (2021a), Figshare 10.6084/m9.figshare.12114408.v1, match
the dataset's electrodes: one row per electrode, in the order of the electrodes table, and the
Gaussian weights agree with the dataset's electrode positions." Rows are matched by index. The
matrix's own ROI (largest Gaussian value over the left AAL regions, reading O5 c) is compared with
the ROI nearest to the dataset's position mirrored to the left hemisphere.

In [ ]:
link = []
for p in P:
    dens, W = PROJ[p]
    el = META[(p, DAYS[p][0][0])][1]
    names = np.array(W.columns)
    L = np.array([c.endswith("_L") for c in names])
    xyz = el[["x", "y", "z"]].values.copy()
    xyz[:, 0] = -np.abs(xyz[:, 0])
    Cl = CEN.values[L]
    dl = dens.values[L].astype(float)
    n = min(len(W), len(el))
    g = W.values[:n][:, L] * dl[None]
    d2 = ((xyz[:n, None, :] - Cl[None]) ** 2).sum(-1)
    d2[:, dl <= 0] = np.inf
    agree = (g.argmax(1) == d2.argmin(1)).mean()
    ok = (g > 0) & np.isfinite(d2)
    r = np.corrcoef(np.log(g[ok]), -d2[ok])[0, 1]
    shifts = {s: (W.values[s:s + len(el)][:, L] * dl).argmax(1) for s in range(len(W) - len(el) + 1)} if len(W) > len(el) else {}
    sh = {s: round(float((v == d2.argmin(1)[: len(v)]).mean()), 3) for s, v in shifts.items()}
    a8 = A.roi_assignment(dens, W, el[["x", "y", "z"]].values, CEN, "c")
    b8 = A.roi_assignment(dens, W, el[["x", "y", "z"]].values, CEN, "b")
    link.append(dict(participant=p, matrix_rows=len(W), electrodes=len(el), roi_agreement=round(agree, 3),
                     corr_logweight_negd2=round(r, 3), eight_roi_label_agreement=round(float((a8[a8 >= 0] == b8[a8 >= 0]).mean()), 3),
                     agreement_by_row_shift=sh))
print(pd.DataFrame(link).to_string())
print("Density row (left hemisphere) of the eight ROIs:")
print(pd.DataFrame({p: PROJ[p][0][[r + "_L" for r in A.ROIS]].astype(float).round(2).values for p in P}, index=[SHORT[r] for r in A.ROIS]).to_string())

**Result: partially reproduced.** Four of the five matrices have one row per electrode; participant
01's has 98 rows for 94 electrodes, and an index match agrees best with no shift (0.628, against
0.394 or less when shifted). The matrix's own ROI and the ROI nearest to the dataset's mirrored
position agree for 51 to 64% of electrodes (69 to 77% among electrodes the matrix puts in one of the
eight ROIs), so the matrices were not computed from these positions exactly. Region claims are run
with ROIs from the matrices (O5 a, c) and from the positions (O5 b).

## cohort-selection

"Of the twelve AJILE12 participants, only five (2 male, 3 female) had at least 1 h of recorded neural
activity of the behaviours of interest, and these five were analysed." Minutes are the summed
durations of the epochs labelled Talk or TV over the files (days 3 to 7), with single-label epochs
(main) and with any epoch whose label includes the class (variant); before artifact rejection only
(the signals of the other seven participants are not read).

In [ ]:
rows = []
for s in sorted(assets.subject.unique()):
    ks = [k for k in META if k[0] == s]
    ep = pd.concat([META[k][2] for k in ks])
    ep["dur"] = (ep.stop - ep.start) / 60
    r = dict(subject=s, days=len(ks))
    for c, lab in [("Talk", "Talk"), ("TV", "TV")]:
        r[c + "_single"] = ep[ep.label == lab].dur.sum()
        r[c + "_any"] = ep[ep.label.str.split(", ").apply(lambda l: lab in l)].dur.sum()
    rows.append(r)
coh = pd.DataFrame(rows).round(1)
for lab in ["single", "any"]:
    coh["each_1h_" + lab] = (coh["Talk_" + lab] >= 60) & (coh["TV_" + lab] >= 60)
    coh["both_1h_" + lab] = (coh["Talk_" + lab] + coh["TV_" + lab]) >= 60
print(coh.to_string())
for c in [c for c in coh.columns if c.startswith(("each", "both"))]:
    print(c, "->", list(coh.subject[coh[c]]))
print("Sex and age: the files' subject entries hold only species and subject_id:", list(META[("08", 3)][0].keys())[:2],
      "; the archive's asset metadata lists species only.")

**Result: reproduced.** Under O2 (a), at least 1 h of each of talking and watching TV over days 3 to
7, exactly subjects 01, 02, 04, 05 and 08 qualify with single-label epochs; counting any epoch whose
label includes the class adds 03 and 06; under O2 (b) all twelve qualify. Sex is not in the dataset,
so the 2 male and 3 female split cannot be checked.

## table1-participants

"Table 1 gives, per participant, sex, age, good electrodes and minutes of talking, watching TV and
using electronics; all five have at least 2 h of data for the two states."

In [ ]:
T1 = {"01": (85, 493, 605, 291), "02": (80, 331, 338, 68), "04": (67, 696, 312, 181), "05": (104, 340, 480, 212), "08": (82, 216, 156, 29)}
rows = []
for p in P:
    goods = [int(g.sum()) for _, _, g, _, _ in DAYS[p]]
    inter = int(np.all([g for _, _, g, _, _ in DAYS[p]], axis=0).sum())
    union = len(CH[p])
    rows.append(dict(participant=p, table1=T1[p][0], day3=goods[0], per_day=goods, min_day=min(goods), all_days=inter, any_day=union))
el_t = pd.DataFrame(rows)
print("Good electrodes:")
print(el_t.to_string())
for c in ["day3", "min_day", "all_days", "any_day"]:
    print("  reading %-8s matches %d of 5" % (c, (el_t[c] == el_t.table1).sum()))
rows = []
for an in [None] + list(ART):
    for p in P:
        if an is None:
            bl = A.build_samples(DAYS[p], None)["blocks"]
        else:
            bl = SAMPLES[(an, p)]["blocks"]
        kept = bl[bl.status == "kept"]
        mins = {c: ((kept.stop - kept.start)[kept.cls == c]).sum() / 60 for c in ["Talk", "TV", "Electronics"]}
        rows.append(dict(artifact=an or "none", participant=p, **{k: round(v, 1) for k, v in mins.items()}))
mn = pd.DataFrame(rows)
for c, i in [("Talk", 1), ("TV", 2), ("Electronics", 3)]:
    mn[c + "_t1"] = mn.participant.map(lambda p: T1[p][i])
    mn[c + "_ok"] = (mn[c] - mn[c + "_t1"]).abs() <= 0.1 * mn[c + "_t1"]
mn["n_within_10pct"] = mn[["Talk_ok", "TV_ok", "Electronics_ok"]].sum(1)
mn["talk_tv_ge_120"] = (mn.Talk + mn.TV) >= 120
print("Minutes per class (single-label blocks inside the ECoG), before and after artifact rejection:")
print(mn.to_string())
print(mn.groupby("artifact").agg(within=("n_within_10pct", "sum"), all_ge_120=("talk_tv_ge_120", "all")).to_string())

**Result: not reproduced.** No reading of the good-electrode count matches all five Table 1 values:
the minimum over days matches 4 of 5 (05 has 100, Table 1 104), the first day 3 of 5. No artifact
reading puts all 15 minute values within 10% (at most 4 of 15 under O4 a-i); without rejection or
under rule (b) the minutes exceed Table 1 (01 talking 763.5 against 493), and under rule (a) they
fall below it (01 talking 171.6 to 208.0). Every participant has at least 120 min of talking and
watching TV under every reading. Sex and age are not in the dataset.

## fig2-highest-accuracy

"Across the 40 ROI and band features, the highest participant-wide classification accuracy of
Talking against Watching TV is 69%, for high gamma in the inferior temporal lobe." Accuracy of a
feature is the mean of its fold accuracies over participants (35 when all five have the ROI).

In [ ]:
print("Main reading, mean fold accuracy (ROI columns, band rows):")
print(mat(GM, "acc").round(3).to_string())
print("folds per feature:", GM.n.min(), "to", GM.n.max())
print("range of accuracies %.3f to %.3f" % (GM.acc.min(), GM.acc.max()))
top = GM.sort_values("acc", ascending=False).head(3)
print("top features:", [(SHORT[r], b, round(a, 3)) for r, b, a in zip(top.roi, top.band, top.acc)])
rows = []
for g in GRID:
    G = GROUP[(g, CHANCES[0])]
    best = G.loc[G.acc.idxmax()]
    it = G[(G.roi == "Temporal_Inf") & (G.band == "high_gamma")]
    rows.append(dict(reading=g, best=(SHORT[best.roi], best.band), best_acc=round(best.acc, 3),
                     it_hg=round(float(it.acc.iloc[0]), 3) if len(it) else np.nan))
R2 = pd.DataFrame(rows)
R2["it_hg_highest"] = R2.best.apply(lambda b: b == ("inferior temporal", "high_gamma"))
R2["within_tol"] = (R2.it_hg - 0.69).abs() <= 0.069
print(R2.to_string())
print("readings where IT high gamma is highest: %d; within 62.1 to 75.9%%: %d; both: %d of %d"
      % (R2.it_hg_highest.sum(), R2.within_tol.sum(), (R2.it_hg_highest & R2.within_tol).sum(), len(R2)))

**Result: reproduced, resting on the ROI reading.** In 30 of the 96 pipelines inferior temporal high
gamma has the highest of the 40 accuracies and lies within 62.1 to 75.9% (0.695 to 0.730); all 30
use the electrodes as features (O6 a) and ROIs from O5 (a) or (b). Under the main reading it is
second (0.613, just below the tolerance) after middle temporal high gamma (0.628).

## Significance of features (Figure 2)

Shared by the four claims that follow. Significance: t-test of fold accuracy minus finite chance,
BH FDR at 0.01 over the 40 features, mean above chance (M6, O9).

In [ ]:
print("Main reading, significant features (1) and finite chance level", round(GM.chance.mean(), 3))
print(mat(GM, "sig_acc").astype(int).to_string())
for ch in CHANCES:
    G = GROUP[(MAIN, ch)]
    print("main pipeline, chance", ch, "significant features:", int(G.sig_acc.sum()), "chance level %.3f" % G.chance.mean())

## fig2-rois-three-bands

"The postcentral gyrus, supramarginal gyrus and superior, middle and inferior temporal lobes each have
three or more bands with accuracy significantly above chance."

In [ ]:
FIVE = ["Postcentral", "SupraMarginal", "Temporal_Sup", "Temporal_Mid", "Temporal_Inf"]


def nsig(G):
    return G[G.sig_acc].groupby("roi").size().reindex(A.ROIS).fillna(0).astype(int)


print("main reading, significant bands per ROI:", {SHORT[k]: v for k, v in nsig(GM).items()})
ok = readings_meeting(lambda G, F: all(nsig(G)[r] >= 3 for r in FIVE))
main_ok = show_readings(ok)
ok4 = readings_meeting(lambda G, F: sum(nsig(G)[r] >= 3 for r in FIVE) >= 4)
print("readings where at least four of the five ROIs have 3 bands:", len(ok4))
best = max(((sum(nsig(GROUP[k])[r] >= 3 for r in FIVE), k) for k in GROUP), key=lambda z: z[0])
print("largest number of the five ROIs with 3 or more bands over all readings:", best[0], "at", best[1])

**Result: reproduced, resting on the readings.** The five ROIs each have three or more significant
bands in 14 of 384 readings, all with the artifact SD over the participant's days (O4 a-i) and a
chance level from all test samples; at least four of the five do in 40. Under the main reading only
the middle temporal lobe has three (postcentral 1, supramarginal 2, inferior temporal 2).

## fig2-inferior-temporal-all-bands

"All five bands in the inferior temporal lobe decode significantly above chance."

In [ ]:
print("main reading, IT significant bands:", int(nsig(GM)["Temporal_Inf"]))
print("IT p-values (main):", GM[GM.roi == "Temporal_Inf"].set_index("band").p_acc.round(5).to_dict())
ok = readings_meeting(lambda G, F: nsig(G)["Temporal_Inf"] == 5)
show_readings(ok)
ok4 = readings_meeting(lambda G, F: nsig(G)["Temporal_Inf"] >= 4)
print("readings with at least 4 IT bands significant:", len(ok4))
print("distribution of IT significant bands over readings:", pd.Series([int(nsig(GROUP[k])["Temporal_Inf"]) for k in GROUP]).value_counts().sort_index().to_dict())

**Result: reproduced, resting on the readings.** All five inferior temporal bands are significant in
134 of 384 readings, all with the electrodes as features (O6 a); under the main reading 2 of 5 are
(beta and high gamma, p = 2e-05; the others p 0.003 to 0.010 do not pass the FDR cut). 206 readings
have none.

## fig2-beta-temporal-only

"Beta decodes above chance only in the middle and inferior temporal lobes."

In [ ]:
def beta_sig(G):
    return sorted(G[(G.band == "beta") & G.sig_acc].roi)


print("main reading, ROIs with significant beta:", [SHORT[r] for r in beta_sig(GM)])
ok = readings_meeting(lambda G, F: beta_sig(G) == sorted(["Temporal_Mid", "Temporal_Inf"]))
show_readings(ok)
okp = readings_meeting(lambda G, F: {"Temporal_Mid", "Temporal_Inf"} <= set(beta_sig(G)) and len(beta_sig(G)) == 3)
print("readings with both temporal ROIs and one other:", len(okp))
print("distribution of beta-significant ROI sets:", pd.Series([", ".join(SHORT[r] for r in beta_sig(GROUP[k])) or "none" for k in GROUP]).value_counts().head(8).to_dict())

**Result: reproduced, resting on the readings.** Beta is significant in exactly the middle and
inferior temporal lobes in 29 of 384 readings. Under the main reading it is significant in those two
and in the supramarginal gyrus; in 206 readings beta is significant nowhere.

## fig2-single-band-rois

"The inferior parietal lobe decodes above chance only with high gamma, and the middle frontal lobe
only with theta."

In [ ]:
def sig_bands(G, roi):
    return sorted(G[(G.roi == roi) & G.sig_acc].band)


print("main: inferior parietal", sig_bands(GM, "Parietal_Inf"), "| middle frontal", sig_bands(GM, "Frontal_Mid"))
print("inferior parietal present in the main reading for", FM[FM.roi == "Parietal_Inf"].participant.unique().tolist())
ok = readings_meeting(lambda G, F: sig_bands(G, "Parietal_Inf") == ["high_gamma"] and sig_bands(G, "Frontal_Mid") == ["theta"])
show_readings(ok)
ok1 = readings_meeting(lambda G, F: sig_bands(G, "Parietal_Inf") == ["high_gamma"] or sig_bands(G, "Frontal_Mid") == ["theta"])
print("readings where one of the two holds:", len(ok1))

**Result: reproduced, resting on the readings.** Both parts hold in only 4 of 384 readings (all O5
a-i, artifact rule b, chance from all test samples); one of the two holds in 49. Under the main
reading neither ROI has a significant band, and the inferior parietal lobe exists for only two
participants (02, 05).

## anova-roi-band-effects

"In a two-way ANOVA on accuracy, both ROI and band contribute significantly to decodability (p
reported as 0 for both)."

In [ ]:
def anova(F, typ=2, level="fold"):
    d = F if level == "fold" else F.groupby(["participant", "roi", "band"], as_index=False).acc.mean()
    m = smf.ols("acc ~ C(roi) * C(band)", data=d).fit()
    return anova_lm(m, typ=typ)


for lvl in ["fold", "participant"]:
    for typ in [2, 1, 3]:
        a = anova(FM, typ, lvl)
        print("main, %s level, type %d:" % (lvl, typ), {k: "F=%.2f p=%.2g" % (a.loc[k, "F"], a.loc[k, "PR(>F)"]) for k in a.index if k != "Residual" and k != "Intercept"})
rows = []
for g in GRID:
    for lvl in ["fold", "participant"]:
        a = anova(FOLDS[g], 2, lvl)
        rows.append(dict(reading=g, level=lvl, p_roi=a.loc["C(roi)", "PR(>F)"], p_band=a.loc["C(band)", "PR(>F)"],
                         p_int=a.loc["C(roi):C(band)", "PR(>F)"], F_int=a.loc["C(roi):C(band)", "F"]))
AN = pd.DataFrame(rows)
AN["both_lt_001"] = (AN.p_roi < 0.001) & (AN.p_band < 0.001)
print("type II over the grid: both main effects p < 0.001 in %d of %d (fold level %d of %d)"
      % (AN.both_lt_001.sum(), len(AN), AN[AN.level == "fold"].both_lt_001.sum(), (AN.level == "fold").sum()))
print("p_roi range %.2g to %.2g; p_band range %.2g to %.2g" % (AN.p_roi.min(), AN.p_roi.max(), AN.p_band.min(), AN.p_band.max()))

**Result: reproduced.** Under the main reading (type II, fold accuracies) ROI F = 7.22, p = 1.7e-08
and band F = 7.18, p = 1e-05, both below 0.001; type I gives the same. Type III (p 0.32 and 0.19)
and participant means (p 0.0087 and 0.028) do not. Over the grid both main effects are below 0.001
in 44 of 96 fold-level readings.

## anova-no-interaction

"The interaction between ROI and band has no significant effect on accuracy (p = 0.46)." Judged on
type II sums of squares over the pooled fold accuracies of the main reading.

In [ ]:
a = anova(FM, 2, "fold")
print("main, type II, fold accuracies: interaction F = %.3f, df = %d, p = %.3g" % (a.loc["C(roi):C(band)", "F"], a.loc["C(roi):C(band)", "df"], a.loc["C(roi):C(band)", "PR(>F)"]))
print("over the grid (type II): interaction p >= 0.05 in %d of %d fold-level readings; p range %.3g to %.3g"
      % ((AN[AN.level == "fold"].p_int >= 0.05).sum(), (AN.level == "fold").sum(), AN[AN.level == "fold"].p_int.min(), AN[AN.level == "fold"].p_int.max()))
print("participant level: p >= 0.05 in %d of %d" % ((AN[AN.level == "participant"].p_int >= 0.05).sum(), (AN.level == "participant").sum()))

**Result: reproduced, resting on an assumption** (type II sums of squares on fold accuracies, the
claims file's choice). The interaction has F = 0.779 on 28 df, p = 0.789. It is not significant in
77 of 96 fold-level readings (p 0.0063 to 0.988) and in all 96 on participant means.

## posthoc-band-averages

"Averaged over ROIs, high gamma has the highest accuracy (63%) and beta the lowest (56%, barely above
the finite chance level)." O11 (a) mean of all fold accuracies of the band; (b) mean of the
per-feature means.

In [ ]:
def band_avgs(G, F, how):
    if how == "a":
        return F.groupby("band").acc.mean()
    return G.groupby("band").acc.mean()


rows = []
for g in GRID:
    for ch in CHANCES:
        G, F = GROUP[(g, ch)], FOLDS[g]
        chance = G.chance.mean()
        for how in "ab":
            b = band_avgs(G, F, how)
            ok = (abs(b["high_gamma"] - 0.63) <= 0.063 and abs(b["beta"] - 0.56) <= 0.056 and b.idxmax() == "high_gamma"
                  and b.idxmin() == "beta" and b["beta"] > chance and (b["beta"] - chance) < (b["high_gamma"] - chance))
            order_ok = b.idxmax() == "high_gamma" and b.idxmin() == "beta"
            rows.append(dict(reading=g, chance=ch, O11=how, hg=b["high_gamma"], beta=b["beta"], top=b.idxmax(), bottom=b.idxmin(), chance_level=chance, ok=ok, order_ok=order_ok))
BA = pd.DataFrame(rows)
print("main (O11 a):", band_avgs(GM, FM, "a").round(3).to_dict(), "chance %.3f" % GM.chance.mean())
print("main (O11 b):", band_avgs(GM, FM, "b").round(3).to_dict())
print("criterion met in %d of %d readings; order (HG top, beta bottom) in %d" % (BA.ok.sum(), len(BA), BA.order_ok.sum()))
print("highest band over readings:", BA.top.value_counts().to_dict(), "| lowest:", BA.bottom.value_counts().to_dict())
print("HG average range %.3f to %.3f; beta %.3f to %.3f" % (BA.hg.min(), BA.hg.max(), BA.beta.min(), BA.beta.max()))

**Result: reproduced, resting on the readings.** The criterion holds in 12 of 768 readings. Under
the main reading high gamma is highest (0.580, within tolerance) and beta (0.549) is within
tolerance and above the chance level 0.520, but theta (0.538) and alpha (0.541) are lower than beta,
so the order fails. High gamma is the top band in 728 of 768 readings; beta is the bottom band in
136.

## posthoc-roi-averages

"Averaged over bands, the inferior temporal lobe has the highest accuracy (66%) and is statistically
higher than every other ROI; the middle frontal lobe is lowest (54%)." Post-hoc tests O12 (a) Tukey
HSD at 0.05 on the pooled fold accuracies; (b) pairwise two-sample t-tests, BH FDR at 0.01.

In [ ]:
def roi_posthoc(F, how):
    others = [r for r in A.ROIS if r != "Temporal_Inf" and r in set(F.roi)]
    if how == "a":
        tk = pairwise_tukeyhsd(F.acc.values, F.roi.values, alpha=0.05)
        t = pd.DataFrame(tk.summary().data[1:], columns=tk.summary().data[0])
        sig = []
        for r in others:
            row = t[((t.group1 == r) & (t.group2 == "Temporal_Inf")) | ((t.group2 == r) & (t.group1 == "Temporal_Inf"))]
            sig.append(bool(row.reject.iloc[0]))
        return sig, len(others)
    ps = [stats.ttest_ind(F[F.roi == "Temporal_Inf"].acc, F[F.roi == r].acc).pvalue for r in others]
    return list(A.bh_fdr(ps, 0.01)), len(others)


rows = []
for g in GRID:
    F = FOLDS[g]
    G = GROUP[(g, CHANCES[0])]
    for o11 in "ab":
        r = F.groupby("roi").acc.mean() if o11 == "a" else G.groupby("roi").acc.mean()
        for o12 in "ab":
            sig, n = roi_posthoc(F, o12)
            rows.append(dict(reading=g, O11=o11, O12=o12, it=r.get("Temporal_Inf", np.nan), mf=r.get("Frontal_Mid", np.nan), top=r.idxmax(), bottom=r.idxmin(),
                             n_sig=sum(sig), n_others=n))
RA = pd.DataFrame(rows)
RA["order_ok"] = (RA.top == "Temporal_Inf") & (RA.bottom == "Frontal_Mid")
RA["ok"] = RA.order_ok & ((RA.it - 0.66).abs() <= 0.066) & ((RA.mf - 0.54).abs() <= 0.054) & (RA.n_sig == 7)
print("main:", FM.groupby("roi").acc.mean().rename(SHORT).round(3).to_dict())
s, n = roi_posthoc(FM, "a")
print("main Tukey: IT significantly different from %d of %d other ROIs; t-tests: %d" % (sum(s), n, sum(roi_posthoc(FM, "b")[0])))
print("criterion met in %d of %d readings; order holds in %d; IT different from all seven in %d"
      % (RA.ok.sum(), len(RA), RA.order_ok.sum(), (RA.n_sig == 7).sum()))
print("top ROI over readings:", RA.top.map(SHORT).value_counts().to_dict(), "| bottom:", RA.bottom.map(SHORT).value_counts().to_dict())
print("IT average range %.3f to %.3f; middle frontal %.3f to %.3f" % (RA.it.min(), RA.it.max(), RA.mf.min(), RA.mf.max()))

**Result: reproduced, resting on the readings.** The criterion holds in 64 of 384 readings. Under
the main reading middle frontal is lowest (0.516, within tolerance) and inferior temporal highest
(0.582, below the 59.4% bound) but significantly different from only 3 of 7 ROIs. The inferior
temporal lobe is the top ROI in 200 readings and differs from all seven in 112.

## fig3-three-class

"With Using Electronics added as a third class, decoding with all electrodes is above the finite
chance level for all four participants with enough data, with high variability; Participant 2 is
highest (above 70% in every band) and much higher than Participant 4." Participants 01, 02, 04, 05;
the features are all electrodes of the eight ROIs; scikit-learn LDA (shared covariance); three
balanced classes; the 7-fold scheme of M5; chance for c = 3 (O16).

In [ ]:
rng3 = np.random.default_rng(303)
rows = []
for an in ["a-ii", "b-ii"]:
    for o5, dr in [("c", "ii"), ("a", "ii"), ("b", "i")]:
        for o7 in "ab":
            for p in ["01", "02", "04", "05"]:
                S = SAMPLES[(an, p)]
                pos = sorted({i for v in roi_sets(p, o5, dr, S["channels"]).values() for i in v})
                sp = A.make_splits(S["cls"], ["Talk", "TV", "Electronics"], o7, rng3)
                for bi, band in enumerate(A.BAND_NAMES):
                    F = S["X"][:, pos, bi].astype(float)
                    accs, nts = [], []
                    for tr, te in sp:
                        clf = LinearDiscriminantAnalysis().fit(A.zscore(F[tr]), S["cls"][tr])
                        accs.append((clf.predict(A.zscore(F[te])) == S["cls"][te]).mean())
                        nts.append(len(te))
                    rows.append(dict(artifact=an, O5=o5 + "-" + dr, O7=o7, participant=p, band=band, acc=np.mean(accs),
                                     chance_fold05=np.mean([A.finite_chance(n, 3, 0.05) for n in nts]),
                                     chance_total05=A.finite_chance(sum(nts), 3, 0.05), chance_total01=A.finite_chance(sum(nts), 3, 0.01),
                                     n_test=sum(nts), electrodes=len(pos)))
TC = pd.DataFrame(rows)
main3 = TC[(TC.artifact == "a-ii") & (TC.O5 == "c-ii") & (TC.O7 == "a")]
print(main3.pivot(index="participant", columns="band", values="acc").round(3).to_string())
print("chance (per fold, 0.05):", main3.groupby("participant").chance_fold05.mean().round(3).to_dict())
res3 = []
for k, g in TC.groupby(["artifact", "O5", "O7"]):
    m = g.pivot(index="participant", columns="band", values="acc")
    above = all((g.acc > g[c]).all() for c in ["chance_fold05"])
    above01 = (g.acc > g.chance_total01).all()
    p2_70 = (m.loc["02"] > 0.70).all()
    p2_top = (m.mean(1).idxmax() == "02")
    p2_gt4 = (m.loc["02"] > m.loc["04"]).all()
    p2_gt4_b = (m.loc["02"] > m.loc["05"]).all()
    res3.append(dict(reading=k, above_all=above, above_all_001=above01, p2_above70=p2_70, p2_highest=p2_top, p2_gt_p4=p2_gt4, p2_gt_t1_5=p2_gt4_b,
                     p2_min=m.loc["02"].min(), best=m.mean(1).idxmax()))
R3 = pd.DataFrame(res3)
R3["ok"] = R3.above_all & R3.p2_above70 & R3.p2_highest & R3.p2_gt_p4
print(R3.round(3).to_string())
print("criterion met in %d of %d readings; above chance everywhere in %d" % (R3.ok.sum(), len(R3), R3.above_all.sum()))

**Result: partially reproduced.** In 10 of 12 readings every participant is above its per-fold
finite chance level in every band (main reading: participant 04 falls below it in low and high
gamma, 0.357 and 0.359 against 0.403). Participant 02 is highest in 11 of 12 and above participant
04 in every band in 9, but its lowest band never exceeds 70% (0.463 to 0.569).

## fig4-not-single-participant

"Every participant has accuracy significantly above chance for multiple ROI and band features." O13
(a) per participant, t-test of the 7 fold accuracies against the per-fold chance, BH FDR at 0.01 over
the participant's features; (b) mean fold accuracy above the participant's finite chance threshold
(n = all test samples).

In [ ]:
def per_participant(F, alpha=0.05, how="a"):
    out = {}
    for p, g in F.groupby("participant"):
        rows = []
        for (roi, band), h in g.groupby(["roi", "band"]):
            if how == "a":
                pv = stats.ttest_1samp(h.acc - h[f"chance_fold_{alpha}"], 0).pvalue
                rows.append(dict(roi=roi, band=band, acc=h.acc.mean(), p=pv, above=h.acc.mean() > h[f"chance_fold_{alpha}"].mean()))
            else:
                rows.append(dict(roi=roi, band=band, acc=h.acc.mean(), p=np.nan, above=h.acc.mean() > A.finite_chance(int(h.ntest.sum()), 2, alpha)))
        d = pd.DataFrame(rows)
        d["sig"] = (A.bh_fdr(d.p, 0.01) & d.above) if how == "a" else d.above
        out[p] = d
    return out


rows = []
for g in GRID:
    for alpha in (0.05, 0.01):
        for how in "ab":
            pp = per_participant(FOLDS[g], alpha, how)
            rows.append(dict(reading=g, alpha=alpha, O13=how, **{p: int(pp[p].sig.sum()) if p in pp else 0 for p in P}))
R4 = pd.DataFrame(rows)
R4["n_ge2"] = (R4[P] >= 2).sum(1)
print("main (O13 a, alpha 0.05): significant features per participant", R4.iloc[0][P].to_dict())
print(R4.groupby(["O13", "alpha"])[P].median().to_string())
print("readings with all five participants >= 2 features: %d of %d; >= 4 participants: %d" % ((R4.n_ge2 == 5).sum(), len(R4), (R4.n_ge2 >= 4).sum()))

**Result: reproduced, resting on the reading of participant-level significance.** Under O13 (b),
mean fold accuracy above the participant's finite chance level, every participant has many features
above chance (medians 17.5 to 26 per participant) in all 192 such readings. Under O13 (a), a t-test
of the 7 fold accuracies with FDR, the median count is 0 to 2 per participant, so the claim fails
there.

## fig4-missing-rois

"Participants 2 and 3 have no electrodes in at least one of the eight ROIs." Under O1 (a), figure
participants 2 and 3 are Table 1's 02 and 04.

In [ ]:
rows = []
for o5 in "cab":
    for dr in ["ii", "i"]:
        miss = {p: [SHORT[r] for r in A.ROIS if r not in roi_sets(p, o5, dr, SAMPLES[("a-ii", p)]["channels"])] for p in P}
        lack = sorted(p for p in P if miss[p])
        rows.append(dict(O5=o5 + "-" + dr, lacking=lack, ok=lack == ["02", "04"], partial={"02", "04"} <= set(lack) and lack != ["02", "04"], missing=miss))
R4m = pd.DataFrame(rows)
print(R4m.to_string())
print("criterion met under %d of %d ROI readings" % (R4m.ok.sum(), len(R4m)))

**Result: partially reproduced.** Participants 02 and 04 (figure 2 and 3 under O1 a) lack at least
one ROI under every ROI reading, but so do 08 (inferior parietal) under all six and 01 under five,
so no reading has exactly those two.

## fig4-participant2-high

"Participant 2 performs best, with several features above 75% accuracy and low and high gamma in the
inferior temporal lobe above 80%."

In [ ]:
rows = []
for g in GRID:
    pm = FOLDS[g].groupby(["participant", "roi", "band"]).acc.mean().reset_index()
    mean_p = pm.groupby("participant").acc.mean()
    p2 = pm[pm.participant == "02"]
    it = p2[p2.roi == "Temporal_Inf"].set_index("band").acc
    rows.append(dict(reading=g, best=mean_p.idxmax(), p2_mean=mean_p["02"], n75=int((p2.acc > 0.75).sum()),
                     it_lg=it.get("low_gamma", np.nan), it_hg=it.get("high_gamma", np.nan), p2_max=p2.acc.max()))
R4h = pd.DataFrame(rows)
R4h["ok"] = (R4h.best == "02") & (R4h.n75 >= 2) & (R4h.it_lg > 0.8) & (R4h.it_hg > 0.8)
R4h["partial"] = (R4h.best == "02") & (R4h.n75 >= 2) & (R4h.it_lg > 0.75) & (R4h.it_hg > 0.75)
print("main:", R4h.iloc[0].to_dict())
print("participant mean accuracy, main:", FM.groupby("participant").acc.mean().round(3).to_dict())
print("best participant over readings:", R4h.best.value_counts().to_dict())
print("criterion met in %d of %d; partial in %d; features above 75%% for participant 02: max %d; IT LG range %.3f to %.3f, IT HG %.3f to %.3f; max feature %.3f"
      % (R4h.ok.sum(), len(R4h), R4h.partial.sum(), R4h.n75.max(), R4h.it_lg.min(), R4h.it_lg.max(), R4h.it_hg.min(), R4h.it_hg.max(), R4h.p2_max.max()))

**Result: partially reproduced.** Participant 02 has the highest mean accuracy under the main
reading (0.607) and in 62 of 96 pipelines, with up to 7 features above 75% (2 under the main
reading); the first two parts and inferior temporal gamma above 75% hold together in 44 pipelines,
but both inferior temporal gamma values exceed 80% in none (at most 0.800).

## Mean differences (Figure 5)

Talking minus watching TV of the fitted LDA class means (z-scored training data), per fold; t-test
against zero with BH FDR at 0.01; a feature is consistent when this and the accuracy test are both
significant (M8).

In [ ]:
print("main reading, mean difference (35 folds when all participants have the ROI):")
print(mat(GM, "meandiff").round(3).to_string())
print("main reading, consistent features (sign shown):")
print((mat(GM, "consistent").astype(float) * np.sign(mat(GM, "meandiff"))).to_string())
print("range of mean differences %.3f to %.3f; consistent: %d" % (GM.meandiff.min(), GM.meandiff.max(), GM.consistent.sum()))
print("consistent features per reading:", pd.Series([int(GROUP[k].consistent.sum()) for k in GROUP]).describe().round(1).to_dict())

## fig5-gamma-higher-talking

"Low and high gamma activity is higher when a participant is talking than when watching TV."

In [ ]:
def cons(G, bands):
    return G[G.consistent & G.band.isin(bands)]


def gamma_ok(G, F):
    c = cons(G, ["low_gamma", "high_gamma"])
    return len(c) and (c.meandiff > 0).all() and set(c.band) == {"low_gamma", "high_gamma"}


c = cons(GM, ["low_gamma", "high_gamma"])
print("main: consistent gamma features", [(SHORT[r], b, round(m, 3)) for r, b, m in zip(c.roi, c.band, c.meandiff)])
ok = readings_meeting(gamma_ok)
show_readings(ok)
fr = [((cons(GROUP[k], ["low_gamma", "high_gamma"]).meandiff > 0).mean() if len(cons(GROUP[k], ["low_gamma", "high_gamma"])) else np.nan) for k in GROUP]
print("fraction of consistent gamma features positive, over readings: median %.2f, readings with a majority positive %d of %d"
      % (np.nanmedian(fr), np.sum(np.array(fr) > 0.5), len(fr)))

**Result: reproduced, resting on the readings.** In 68 of 384 readings every consistent gamma
feature is positive and both gamma bands have one. Under the main reading only postcentral low gamma
is consistent (+0.196). Where any consistent gamma feature exists, the median share positive is
1.00.

## fig5-theta-alpha-lower-talking

"Theta and alpha activity is lower when a participant is talking than when watching TV."

In [ ]:
def ta_ok(G, F):
    c = cons(G, ["theta", "alpha"])
    return len(c) and (c.meandiff < 0).all() and set(c.band) == {"theta", "alpha"}


c = cons(GM, ["theta", "alpha"])
print("main: consistent theta and alpha features", [(SHORT[r], b, round(m, 3)) for r, b, m in zip(c.roi, c.band, c.meandiff)])
ok = readings_meeting(ta_ok)
show_readings(ok)
fr = [((cons(GROUP[k], ["theta", "alpha"]).meandiff < 0).mean() if len(cons(GROUP[k], ["theta", "alpha"])) else np.nan) for k in GROUP]
print("fraction of consistent theta/alpha features negative: median %.2f, readings with a majority negative %d of %d"
      % (np.nanmedian(fr), np.sum(np.array(fr) > 0.5), len(fr)))

**Result: reproduced, resting on the readings.** In 30 of 384 readings, all with O4 a-i, every
consistent theta and alpha feature is negative and both bands have one; under the main reading none
is consistent, though the mean differences of alpha are negative in all eight ROIs.

## fig5-pre-postcentral

"The largest differences are in the precentral and postcentral gyri, with consistent differences in
theta, alpha, low gamma and high gamma."

In [ ]:
SIGNS = {"theta": -1, "alpha": -1, "low_gamma": 1, "high_gamma": 1}


def prepost(G):
    top = G.loc[G.meandiff.abs().idxmax()]
    n = 0
    for r in ["Precentral", "Postcentral"]:
        for b, s in SIGNS.items():
            row = G[(G.roi == r) & (G.band == b)]
            n += int(len(row) and row.consistent.iloc[0] and np.sign(row.meandiff.iloc[0]) == s)
    return top.roi in ("Precentral", "Postcentral"), n, (SHORT[top.roi], top.band, round(top.meandiff, 3))


print("main:", prepost(GM))
ok = readings_meeting(lambda G, F: prepost(G)[0] and prepost(G)[1] == 8)
show_readings(ok)
ns = pd.Series([prepost(GROUP[k])[1] for k in GROUP])
print("of the 8 features consistent with the expected sign, over readings:", ns.value_counts().sort_index().to_dict())
print("largest |difference| in pre- or postcentral in %d of %d readings" % (sum(prepost(GROUP[k])[0] for k in GROUP), len(GROUP)))

**Result: reproduced, resting on the readings.** Both parts hold in 2 of 384 readings. The largest
absolute difference is in the pre- or postcentral gyrus in 348 readings, but the eight features are
consistent with the expected signs in only 6 readings (6 or more in 11). Under the main reading the
largest difference is middle frontal low gamma (+0.202) and 1 of the 8 is consistent.

## fig5-parietal-temporal-no-gamma

"In the supramarginal gyrus, inferior parietal lobe and superior temporal lobe there is no consistent
gamma increase during talking, while the theta and alpha differences are still present there."

In [ ]:
THREE = ["SupraMarginal", "Parietal_Inf", "Temporal_Sup"]


def ptg(G):
    g = G[G.roi.isin(THREE) & G.band.isin(["low_gamma", "high_gamma"])]
    gamma_none = not (g.consistent & (g.meandiff > 0)).any()
    ta = {r: bool((G[(G.roi == r) & G.band.isin(["theta", "alpha"])].pipe(lambda d: d.consistent & (d.meandiff < 0))).any()) for r in THREE}
    return gamma_none, ta


print("main:", ptg(GM))
print(GM[GM.roi.isin(THREE)][["roi", "band", "meandiff", "md_lo", "md_hi", "p_md", "consistent"]].round(4).to_string())
ok = readings_meeting(lambda G, F: ptg(G)[0] and all(ptg(G)[1].values()))
show_readings(ok)
okp = readings_meeting(lambda G, F: ptg(G)[0] and sum(ptg(G)[1].values()) >= 2)
print("gamma part holds and theta/alpha in at least 2 of 3 ROIs:", len(okp), "| gamma part alone:", len(readings_meeting(lambda G, F: ptg(G)[0])))

**Result: partially reproduced.** No gamma feature of the three ROIs is a consistent increase in 352
of 384 readings (main reading included), but theta or alpha is consistent in all three ROIs in no
reading; in two of the three in 22. Under the main reading the alpha differences are negative there
(-0.098 to -0.155) without passing the FDR cut.

## fig5-middle-temporal-tv

"The middle temporal lobe shows consistently higher activity in 4 of the 5 bands when participants
watch TV than when they talk."

In [ ]:
def mt_neg(G):
    g = G[G.roi == "Temporal_Mid"]
    return int((g.consistent & (g.meandiff < 0)).sum())


print("main: middle temporal bands consistent with a negative difference:", mt_neg(GM))
print(GM[GM.roi == "Temporal_Mid"][["band", "acc", "sig_acc", "meandiff", "p_md", "consistent"]].round(4).to_string())
ok = readings_meeting(lambda G, F: mt_neg(G) == 4)
show_readings(ok)
print("distribution over readings:", pd.Series([mt_neg(GROUP[k]) for k in GROUP]).value_counts().sort_index().to_dict())

**Result: partially reproduced.** Four middle temporal bands consistent with a negative difference
occur in no reading; three occur in 19 of 384. Under the main reading none is consistent (beta, low
and high gamma pass the accuracy test; the mean differences of alpha and beta, -0.107 and -0.106
with p = 0.007 and 0.008, do not pass the FDR cut).

## inferior-temporal-high-gamma-inconsistent

"High gamma in the inferior temporal lobe decodes well but its mean difference is virtually zero."
A claim of absence judged on the paper's test (M8) under the main reading; the grid is a variant.

In [ ]:
row = GM[(GM.roi == "Temporal_Inf") & (GM.band == "high_gamma")].iloc[0]
print("main: IT high gamma accuracy %.3f (significant %s, p = %.3g); mean difference %.4f, 95%% CI %.4f to %.4f, p = %.3g, significant after BH: %s"
      % (row.acc, row.sig_acc, row.p_acc, row.meandiff, row.md_lo, row.md_hi, row.p_md, row.sig_md))
rank = (GM.meandiff.abs() >= abs(row.meandiff)).sum()
print("its |mean difference| ranks %d of %d features (median |difference| %.4f)" % (rank, len(GM), GM.meandiff.abs().median()))
v = pd.DataFrame([dict(sig_acc=GROUP[k].query("roi=='Temporal_Inf' and band=='high_gamma'").sig_acc.iloc[0],
                       sig_md=GROUP[k].query("roi=='Temporal_Inf' and band=='high_gamma'").sig_md.iloc[0]) for k in GROUP])
print("over readings: accuracy significant in %d, mean difference significant in %d, of %d" % (v.sig_acc.sum(), v.sig_md.sum(), len(v)))

**Result: reproduced.** Under the main reading inferior temporal high gamma is significant for
accuracy (0.613, p = 2.4e-05) and its mean difference (0.032, 95% CI -0.059 to 0.122, p = 0.48) is
not; its absolute difference ranks 33 of 40. The mean difference is not significant in any of the
384 readings.

## all-rois-but-inferior-parietal

"Every ROI except the inferior parietal lobe has at least one band that both separates the two
behaviours and is consistent across participants."

In [ ]:
def roi_cons(G):
    return G[G.consistent].groupby("roi").size().reindex(A.ROIS).fillna(0).astype(int)


print("main, consistent bands per ROI:", {SHORT[k]: v for k, v in roi_cons(GM).items()})
SEVEN = [r for r in A.ROIS if r != "Parietal_Inf"]
ok = readings_meeting(lambda G, F: all(roi_cons(G)[r] >= 1 for r in SEVEN) and roi_cons(G)["Parietal_Inf"] == 0)
show_readings(ok)
okp = readings_meeting(lambda G, F: sum(roi_cons(G)[r] >= 1 for r in SEVEN) >= 6 and roi_cons(G)["Parietal_Inf"] == 0)
print("six of seven with IP none:", len(okp))
print("number of the seven ROIs with a consistent band, over readings:", pd.Series([sum(roi_cons(GROUP[k])[r] >= 1 for r in SEVEN) for k in GROUP]).value_counts().sort_index().to_dict())

**Result: reproduced, resting on the readings.** It holds in 2 of 384 readings (and six of seven in
10); in 250 readings no ROI has a consistent band, and under the main reading only the postcentral
gyrus does.

## fig6-sign-consistency

"The sign of the Talking minus Watching TV difference is the same in all participants for 9 of the 40
ROI and band combinations (22.5%)." Per participant, the mean over the 7 folds of the ROI's mean
difference. O14 (a) the participants that have the ROI share a sign; (b) only combinations present in
all five can count.

In [ ]:
def same_sign(F, how):
    m = F.groupby(["roi", "band", "participant"]).meandiff.mean().unstack()
    out = []
    for (roi, band), r in m.iterrows():
        v = r.dropna()
        if how == "b" and len(v) < 5:
            continue
        if len(v) >= 2 and (np.all(v > 0) or np.all(v < 0)):
            out.append((roi, band, int(np.sign(v.iloc[0]))))
    return out


for how in "ab":
    s = same_sign(FM, how)
    print("main, O14 %s: %d combinations" % (how, len(s)), [(SHORT[r], b, sg) for r, b, sg in s])
rows = []
for g in GRID:
    for how in "ab":
        rows.append(dict(reading=g, O14=how, n=len(same_sign(FOLDS[g], how))))
R6 = pd.DataFrame(rows)
print("count over readings:", R6.n.value_counts().sort_index().to_dict())
print("readings with 8 to 10: %d of %d; 7 or 11: %d" % (R6.n.between(8, 10).sum(), len(R6), R6.n.isin([7, 11]).sum()))
print("main per-participant matrix of mean differences (participant 02):")
print(FM[FM.participant == "02"].groupby(["band", "roi"]).meandiff.mean().unstack().rename(columns=SHORT).round(3).to_string())

**Result: reproduced, resting on the readings.** 8 to 10 combinations share a sign in 49 of 192
readings (counts range 0 to 14). Under the main reading O14 (a) gives 7 and O14 (b) gives 2.

## fig6-examples

"High gamma in the precentral, middle frontal and postcentral regions, and alpha in the precentral,
inferior parietal and supramarginal regions, have the same sign in all participants."

In [ ]:
SIX = [("Precentral", "high_gamma"), ("Frontal_Mid", "high_gamma"), ("Postcentral", "high_gamma"),
       ("Precentral", "alpha"), ("Parietal_Inf", "alpha"), ("SupraMarginal", "alpha")]


def six_count(F, how):
    s = {(r, b) for r, b, _ in same_sign(F, how)}
    return sum(x in s for x in SIX)


print("main:", {how: six_count(FM, how) for how in "ab"})
m = FM.groupby(["roi", "band", "participant"]).meandiff.mean().unstack()
print(m.loc[SIX].round(3).rename(index=SHORT).to_string())
R6["six"] = [six_count(FOLDS[g], how) for g in GRID for how in "ab"]
print("over readings:", R6.six.value_counts().sort_index().to_dict(), "| all six in", (R6.six == 6).sum(), "| 4 or 5 in", R6.six.isin([4, 5]).sum())

**Result: partially reproduced.** All six named combinations never share a sign in one reading; 4 or
5 do in 44 of 192 (main reading 4 under O14 a). Precentral high gamma fails through participant 08
(-0.054) and supramarginal alpha through participants 02 and 04.

## Figure 7: adding ROIs

Per participant and band, ROIs ranked and added one at a time. O15 (a): within each outer fold one
training fold (the one next to the buffer) is held out as a validation set to rank ROIs by their
single-ROI accuracy, the classifier is trained on the other training folds and tested on the test
fold; (b): ROIs ranked by the participant's single-ROI cross-validated accuracies, and the 7-fold
scheme of M5. Run on the main artifact reading for every ROI reading, O6 and O7.

In [ ]:
rng7 = np.random.default_rng(707)


def curves(S, rois, o6, o7, o15, single=None):
    y = S["cls"]
    use = np.where(np.isin(y, ["Talk", "TV"]))[0]
    folds, plan = A.folds_and_buffers(len(use), o7)
    names = list(rois)
    out = {}
    for bi, band in enumerate(A.BAND_NAMES):
        feats = {r: (S["X"][:, rois[r], bi].astype(float) if o6 == "a" else S["X"][:, rois[r], bi].astype(float).mean(1, keepdims=True)) for r in names}
        accs = np.zeros((len(plan), len(names)))
        for fi, (train, i) in enumerate(plan):
            te = A.balance(use[folds[i]], y, rng7)
            if o15 == "a":
                v = [j for j in train if abs(j - i) == min(abs(t - i) for t in train)][-1]
                tr = A.balance(use[np.concatenate([folds[j] for j in train if j != v])], y, rng7)
                va = A.balance(use[folds[v]], y, rng7)
                sc = []
                for r in names:
                    pr, _, _ = A.lda2(A.zscore(feats[r][tr]), y[tr] == "Talk", A.zscore(feats[r][va]))
                    sc.append((pr == (y[va] == "Talk")).mean())
                rank = [names[k] for k in np.argsort(sc)[::-1]]
            else:
                tr = A.balance(use[np.concatenate([folds[j] for j in train])], y, rng7)
                rank = sorted(names, key=lambda r: -single[(r, band)])
            for k in range(1, len(names) + 1):
                Fk = np.concatenate([feats[r] for r in rank[:k]], axis=1)
                pr, _, _ = A.lda2(A.zscore(Fk[tr]), y[tr] == "Talk", A.zscore(Fk[te]))
                accs[fi, k - 1] = (pr == (y[te] == "Talk")).mean()
        out[band] = accs
    return out


t = time.time()
CURVES = {}
for g in GRID:
    an, o5, dr, o6, o7 = g
    if an != "a-ii":
        continue
    single_all = FOLDS[g].groupby(["participant", "roi", "band"]).acc.mean()
    for o15 in "ab":
        for p in P:
            S = SAMPLES[(an, p)]
            rois = roi_sets(p, o5, dr, S["channels"])
            CURVES[(g, o15, p)] = curves(S, rois, o6, o7, o15, single_all.loc[p])
print(len(CURVES), "curve sets in", round(time.time() - t), "s")


def curve_table(g, o15):
    rows = []
    for p in P:
        for band, accs in CURVES[(g, o15, p)].items():
            m = accs.mean(0)
            ci = stats.t.ppf(0.975, accs.shape[0] - 1) * stats.sem(accs[:, 0])
            rows.append(dict(participant=p, band=band, k=len(m), top1=m[0], all=m[-1], best=m.max(), best_k=int(m.argmax()) + 1,
                             ci_top1=ci, curve=np.round(m, 3)))
    return pd.DataFrame(rows)


CT = {(g, o15): curve_table(g, o15) for g in GRID if g[0] == "a-ii" for o15 in "ab"}
ct = CT[(MAIN, "a")]
print("main reading, O15 (a):")
print(ct.drop(columns="curve").round(3).to_string())
print("curves (mean test accuracy for k = 1, 2, ... ROIs), participant 01:")
for r in ct[ct.participant == "01"].itertuples():
    print("  ", r.band, r.curve)
print("range of curve values %.3f to %.3f" % (min(c.min() for c in ct.curve), max(c.max() for c in ct.curve)))

## fig7-non-additive

"Adding electrodes from further top-ranked ROIs generally does not improve a participant's decoding
performance." Readings: (a) additive when all ROIs beat the top ROI alone; (b) additive when the best
over k beats the top ROI by more than the 95% interval of the top ROI's fold accuracies.

In [ ]:
rows = []
for (g, o15), c in CT.items():
    na = int((c["all"] > c.top1).sum())
    nb = int((c.best - c.top1 > c.ci_top1).sum())
    rows.append(dict(reading=g, O15=o15, additive_a=na, additive_b=nb))
R7 = pd.DataFrame(rows)
print(R7.to_string())
print("main (O15 a): additive curves, reading (a) %d of 25, reading (b) %d of 25" % (R7.iloc[0].additive_a, R7.iloc[0].additive_b))
print("readings with at most 5 additive: (a) %d, (b) %d, of %d; 6 to 12: (a) %d, (b) %d"
      % ((R7.additive_a <= 5).sum(), (R7.additive_b <= 5).sum(), len(R7), R7.additive_a.between(6, 12).sum(), R7.additive_b.between(6, 12).sum()))

**Result: reproduced, resting on the reading of additive.** Under reading (b) at most 5 of 25 curves
are additive in 36 of 48 readings (main reading: 8). Under reading (a) all ROIs beat the top ROI in
22 of 25 curves under the main reading, and at most 5 in only 2 readings.

## fig7-participant4-low-gamma

"The only notable additive case is Participant 4 in the low gamma band." Figure participant 4 is
Table 1's 05 under O1 (a) and 04 under O1 (b).

In [ ]:
rows = []
for (g, o15), c in CT.items():
    c = c.assign(gain=c.best - c.top1).sort_values("gain", ascending=False).reset_index(drop=True)
    for o1, p4 in [("a", "05"), ("b", "04")]:
        pos = c.index[(c.participant == p4) & (c.band == "low_gamma")][0] + 1
        rows.append(dict(reading=g, O15=o15, O1=o1, rank=pos, gain=c.gain[pos - 1], top=(c.participant[0], c.band[0], round(c.gain[0], 3))))
R7b = pd.DataFrame(rows)
print(R7b.head(4).to_string())
print("rank 1 in %d of %d; among top three in %d" % ((R7b["rank"] == 1).sum(), len(R7b), (R7b["rank"] <= 3).sum()))

**Result: partially reproduced.** Figure participant 4's low gamma curve never has the largest gain
of the 25; it is among the three largest in 5 of 96 readings. Under the main reading it ranks 9 (O1
a) or 21 (O1 b), and the largest gain is participant 02's low gamma (0.159).

## fig7-above-70

"With a simple LDA, every participant reaches more than 70% accuracy, with a maximum of about 80% for
Participant 4 in the low and high gamma bands."

In [ ]:
rows = []
for (g, o15), c in CT.items():
    best = c.groupby("participant").best.max()
    for o1, p4 in [("a", "05"), ("b", "04")]:
        lg = c[(c.participant == p4) & (c.band == "low_gamma")].best.iloc[0]
        hg = c[(c.participant == p4) & (c.band == "high_gamma")].best.iloc[0]
        rows.append(dict(reading=g, O15=o15, O1=o1, n_above70=int((best > 0.7).sum()), p4_lg=lg, p4_hg=hg,
                         p4_highest=best.idxmax() == p4, **{"best_" + p: best[p] for p in P}))
R7c = pd.DataFrame(rows)
R7c["ok"] = (R7c.n_above70 == 5) & ((R7c.p4_lg - 0.8).abs() <= 0.08) & ((R7c.p4_hg - 0.8).abs() <= 0.08) & R7c.p4_highest
print(R7c.head(2).round(3).to_string())
print("criterion met in %d of %d; participants above 70%%, over readings:" % (R7c.ok.sum(), len(R7c)), R7c.n_above70.value_counts().sort_index().to_dict())
print("best accuracy per participant, range over readings:", {p: (round(R7c["best_" + p].min(), 3), round(R7c["best_" + p].max(), 3)) for p in P})

**Result: partially reproduced.** Four participants exceed 70% in 18 of 96 readings and all five in
2, where figure participant 4 is not the highest or not near 80%; under the main reading two do (02
0.798, 05 0.714). Best accuracies range 0.546 to 0.722 (01), 0.625 to 0.882 (02), 0.592 to 0.735
(04), 0.626 to 0.766 (05), 0.558 to 0.703 (08).

## discussion-above-80

"Subject-specific classification performance regularly exceeds 80% with a simple LDA." Read as at
least three of the five participants with some feature (Figure 4) or curve point (Figure 7) above 80%.

In [ ]:
rows = []
for (g, o15), c in CT.items():
    f4 = FOLDS[g].groupby(["participant", "roi", "band"]).acc.mean().groupby("participant").max()
    f7 = c.groupby("participant").best.max()
    top = pd.concat([f4, f7], axis=1).max(1)
    rows.append(dict(reading=g, O15=o15, n_above80=int((top > 0.8).sum()), max_any=top.max(), **{p: round(top[p], 3) for p in P}))
R8 = pd.DataFrame(rows)
print(R8.head(2).to_string())
print("participants above 80% over readings:", R8.n_above80.value_counts().sort_index().to_dict(), "| max %.3f" % R8.max_any.max())

**Result: partially reproduced.** At most one participant exceeds 80% in any reading (31 of 48;
maximum 0.882); three never do.

## fig7-exceeds-group

"For individual participants, performance usually exceeds the group-level performance of Figure 2."
(a) each participant's best Figure 7 accuracy against the highest participant-wide accuracy of Figure
2, at least four of five; (b) each curve's best against the participant-wide accuracy of the same
band in the curve's top ROI, more than half of the 25.

In [ ]:
rows = []
for (g, o15), c in CT.items():
    G = GROUP[(g, CHANCES[0])]
    best_p = c.groupby("participant").best.max()
    na = int((best_p > G.acc.max()).sum())
    single = FOLDS[g].groupby(["participant", "roi", "band"]).acc.mean()
    nb = 0
    for r in c.itertuples():
        sub = single.loc[r.participant].xs(r.band, level="band")
        top_roi = sub.idxmax()
        ga = G[(G.roi == top_roi) & (G.band == r.band)].acc.iloc[0]
        nb += int(r.best > ga)
    rows.append(dict(reading=g, O15=o15, group_max=G.acc.max(), n_part_a=na, n_curves_b=nb))
R9 = pd.DataFrame(rows)
print(R9.head(2).round(3).to_string())
print("reading (a) at least 4 of 5 in %d of %d; reading (b) more than 12 of 25 in %d" % ((R9.n_part_a >= 4).sum(), len(R9), (R9.n_curves_b > 12).sum()))

**Result: reproduced.** Under reading (a) at least four of five participants' best curve accuracies
exceed the highest participant-wide accuracy in 39 of 48 readings (main: 4 of 5, against 0.628), and
under reading (b) more than half of the curves exceed the participant-wide accuracy of their band in
their top ROI in all 48 (main: 20 and 25 of 25).

## fig7-best-band-varies

"The band that gives a participant's highest performance varies by participant; Participant 5 is best
with theta and Participant 3 best with alpha." Under O1 (a), figure 5 is Table 1's 08, figure 3 is 04.

In [ ]:
rows = []
for (g, o15), c in CT.items():
    bb = c.loc[c.groupby("participant").best.idxmax()].set_index("participant").band
    rows.append(dict(reading=g, O15=o15, **bb.to_dict()))
R10 = pd.DataFrame(rows)
R10["ok"] = (R10["08"] == "theta") & (R10["04"] == "alpha")
R10["one"] = (R10["08"] == "theta") | (R10["04"] == "alpha")
print(R10.head(2).to_string())
print("best band of 08:", R10["08"].value_counts().to_dict(), "| of 04:", R10["04"].value_counts().to_dict())
print("both in %d of %d; one in %d; number of distinct best bands across participants (main): %d"
      % (R10.ok.sum(), len(R10), R10.one.sum(), R10.iloc[0][P].nunique()))

**Result: partially reproduced.** The best band does vary (two distinct best bands under the main
reading), but participant 08's best band is never theta when 04's is alpha; one of the two holds in
16 of 48 readings (main O15 a: 04 alpha, 08 alpha).

## Summary

| Claim | Outcome |
|---|---|
| roi-projection-link | partially reproduced |
| cohort-selection | reproduced |
| table1-participants | not reproduced |
| fig2-highest-accuracy | reproduced (assumed) |
| fig2-rois-three-bands | reproduced (assumed) |
| fig2-inferior-temporal-all-bands | reproduced (assumed) |
| fig2-beta-temporal-only | reproduced (assumed) |
| fig2-single-band-rois | reproduced (assumed) |
| anova-roi-band-effects | reproduced |
| anova-no-interaction | reproduced (assumed) |
| posthoc-band-averages | reproduced (assumed) |
| posthoc-roi-averages | reproduced (assumed) |
| fig3-three-class | partially reproduced |
| fig4-not-single-participant | reproduced (assumed) |
| fig4-missing-rois | partially reproduced |
| fig4-participant2-high | partially reproduced |
| fig5-gamma-higher-talking | reproduced (assumed) |
| fig5-theta-alpha-lower-talking | reproduced (assumed) |
| fig5-pre-postcentral | reproduced (assumed) |
| fig5-parietal-temporal-no-gamma | partially reproduced |
| fig5-middle-temporal-tv | partially reproduced |
| inferior-temporal-high-gamma-inconsistent | reproduced |
| all-rois-but-inferior-parietal | reproduced (assumed) |
| fig6-sign-consistency | reproduced (assumed) |
| fig6-examples | partially reproduced |
| fig7-non-additive | reproduced (assumed) |
| fig7-participant4-low-gamma | partially reproduced |
| fig7-above-70 | partially reproduced |
| discussion-above-80 | partially reproduced |
| fig7-exceeds-group | reproduced |
| fig7-best-band-varies | partially reproduced |

"Assumed" marks an outcome that holds under some of the readings the claims file names (its
criteria say "under one reading") but not under all, or that rests on a choice the paper does not
make. Many of these hold in a small share of the readings; each section gives the count.

A failure to reproduce here may be a failure of this attempt, not of the paper: see the reasoning
and the differences from the paper's method.

How the analysis differs from the paper's: the paper's code is not available, so the folds, the
buffer, the chance level, the ROI assignment, the artifact rule's details and the feature of an ROI
are run under the readings the claims file names; the analysed electrodes are those good on every
day; blocks with two or more labels are left out; the Butterworth order (4) and zero phase are a
standard choice, applied in the frequency domain.

What was found about the dataset: in two files (sub-01 ses-7, sub-04 ses-3) the epochs are timed
from 19:00 while the ECoG starts at 03:31 and 03:20, so 223 and 77 labelled blocks lie past the end
of the ECoG unless an offset is applied; with it, sub-01 ses-7 decodes like the other days but
sub-04 ses-3 does not (0.460), so its clock is not established. The good-electrode flags change from
day to day. The files hold no sex or age. 110 of the 3465 class epochs are not about 2 min long
(0.1 to 482 s). The projection matrices agree with the dataset's electrode positions for only about
60% of electrodes, and participant 01's has four rows more than the dataset has electrodes.

In [ ]:
print("runtime %.1f min, peak memory %.2f GB (this process), children %.2f GB"
      % ((time.time() - T0) / 60, resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / 1e6, resource.getrusage(resource.RUSAGE_CHILDREN).ru_maxrss / 1e6))